In [5]:
!pip -q install transformers datasets accelerate evaluate

In [6]:
!pip -q install kagglehub pymorphy3 pymorphy2-dicts-ru razdel

### Задача

**Модерация комментариев в региональном новостном портале**

Функциональные требования:

- Классификация комментариев пользователей на "токсичные/нормальные" в реальном времени

- Обработка 500-1000 комментариев в час в пиковое время

- *Точность* не ниже 75% (с возможностью ручной проверки спорных случаев)

Нефункциональные требования и ограничения:

- Бюджет проекта: 150 тыс. рублей на весь цикл разработки и внедрения

- Инфраструктура: один сервер с 4 ядрами CPU, 8GB RAM, без GPU

- Время отклика: не более 100ms на один комментарий

- SLA 99.5% — нельзя допустить падения модели из-за нехватки ресурсов

- Запрет на передачу данных третьим сторонам (нельзя использовать API внешних LLM)

### Импорты и настройка среды

In [7]:
import gc
import os
import re
import time
import json
import math
import string
import random
import tracemalloc

import numpy as np
import pandas as pd

from dataclasses import dataclass
from functools import lru_cache

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, f1_score, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

import kagglehub

import razdel
import pymorphy3

import torch
import torch.nn.functional as F

import evaluate
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

In [8]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

### Датасет

In [9]:
dataset_name = "blackmoon/russian-language-toxic-comments"
path = kagglehub.dataset_download(dataset_name)
path

Using Colab cache for faster access to the 'russian-language-toxic-comments' dataset.


'/kaggle/input/russian-language-toxic-comments'

In [10]:
!ls {path}

labeled.csv


In [11]:
df = pd.read_csv(path + '/labeled.csv')
df['toxic'] = df['toxic'].astype(int)
df.head()

,comment,toxic
0,"Верблюдов-то за что? Дебилы, бл...\n",1
1,"Хохлы, это отдушина затюканого россиянина, мол...",1
2,Собаке - собачья смерть\n,1
3,"Страницу обнови, дебил. Это тоже не оскорблени...",1
4,"тебя не убедил 6-страничный пдф в том, что Скр...",1


In [12]:
df.isna().sum()

,0
comment,0
toxic,0


In [13]:
df.shape

(14412, 2)

### Базовый EDA

Распределение классов и длин текстов по классам

In [14]:
df["len_chars"] = df["comment"].str.len()
df["len_words"] = df["comment"].str.split().str.len()

In [15]:
df["toxic"].value_counts(normalize=True)

,proportion
toxic,
0,0.66514
1,0.33486


Дубликаты

In [16]:
df["text_norm_simple"] = (
    df["comment"].str.lower()
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)

### Разбиение на train/test

In [17]:
df_f = df.drop_duplicates("text_norm_simple").copy()

X = df_f["comment"].values
y = df_f["toxic"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

len(X_train), len(X_test), y_train.mean(), y_test.mean()

(11324, 2831, np.float64(0.3338926174496644), np.float64(0.33380430943129635))

### Метрики

In [18]:
def eval_binary(y_true, proba, thr=0.5, title=""):
    pred = (proba >= thr).astype(int)
    out = {
        "acc": accuracy_score(y_true, pred),
        "f1": f1_score(y_true, pred),
        "roc_auc": roc_auc_score(y_true, proba),
    }
    print(title, out)
    print(classification_report(y_true, pred, digits=4))
    return out

### Предобработка

In [19]:
morph = pymorphy3.MorphAnalyzer()

URL_RE  = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
USER_RE = re.compile(r"@\w+")
NUM_RE  = re.compile(r"\b\d+([.,]\d+)?\b", re.UNICODE)
WS_RE   = re.compile(r"\s+", re.UNICODE)
WORD_RE = re.compile(r"^[a-zа-яё]+$", re.IGNORECASE)

def normalize_text(s):
    s = s.lower()
    s = URL_RE.sub(" __url__ ", s)
    s = USER_RE.sub(" __user__ ", s)
    s = NUM_RE.sub(" __num__ ", s)
    s = WS_RE.sub(" ", s).strip()
    return s

@lru_cache(maxsize=100_000)
def lemma_token_cached(token):
    if WORD_RE.match(token):
        return morph.parse(token)[0].normal_form
    return token

def preprocess_one_cached(s):
    s = normalize_text(s)
    tokens = [t.text for t in razdel.tokenize(s)]
    lemmas = [lemma_token_cached(t) for t in tokens]
    return " ".join(lemmas)

def preprocess_corpus(texts):
    out = [preprocess_one_cached(str(t)) for t in texts]
    return np.asarray(out, dtype=object)

In [20]:
lr_tfidf = Pipeline(steps=[
    ("vect", TfidfVectorizer(
        sublinear_tf=False,
        ngram_range=(1,2),
        min_df=2,
        max_df=0.5
    )),
    ("clf", LogisticRegression(
        C=float(np.float64(10.0)),
        max_iter=2000,
        n_jobs=-1
    ))
])

X_train_pp = preprocess_corpus(X_train)
X_test_pp  = preprocess_corpus(X_test)

lr_tfidf.fit(X_train_pp, y_train)

proba_lr = lr_tfidf.predict_proba(X_test_pp)[:, 1]
metrics_lr = eval_binary(y_test, proba_lr, title="LogReg+TFIDF")

LogReg+TFIDF {'acc': 0.876368774284705, 'f1': 0.7986191024165707, 'roc_auc': np.float64(0.9373318857412175)}
              precision    recall  f1-score   support

           0     0.8768    0.9475    0.9108      1886
           1     0.8752    0.7344    0.7986       945

    accuracy                         0.8764      2831
   macro avg     0.8760    0.8409    0.8547      2831
weighted avg     0.8763    0.8764    0.8734      2831



### Попробуем transformer

#### Вспоминаем детали

<center><img src ="https://edunet.kea.su/repo/EduNet-content/dev-2.3/L10/out/transformer_architecture.png" width="450"></center>

<center><em>Архитектура трансформера</em></center>

<center><em>Source: <a href="https://arxiv.org/pdf/1706.03762.pdf"> Attention Is All You Need</a></em></center>

#### rubert-tiny2

45 MB, 12M parameters

[Маленький и быстрый BERT для русского языка](https://habr.com/ru/articles/562064/)

rubert-tiny:
- Словарь 30К токенов
- Размер эмбеддинга сокращен с 768 до 312
- Число слоёв уменьшено с 12 до 3
- Эмбеддинги инициализированы из bert-multilingual, все остальные веса – случайным образом
- Обучение с дистилляцией на восьми (!) лоссах
- В качестве обучающей выборки я взял три параллельных корпуса англо-русских предложений: от Яндекс.Переводчика, OPUS-100 и Tatoeba, суммарно 2.5 млн коротких текстов.

rubert-tiny 2:
- Словарь 80К токенов
- Максимальную длина текста увеличена с 512 до 2048 токенов
- Модель дообучена на комбинации задач masked language modelling, natural language inference, и аппроксимации эмбеддингов LaBSE

In [21]:
model_name = "cointegrated/rubert-tiny2"

train_ds = Dataset.from_dict({"text": X_train.tolist(), "label": y_train.tolist()})
test_ds  = Dataset.from_dict({"text": X_test.tolist(),  "label": y_test.tolist()})

tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [22]:
max_length = 128

def tok(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=max_length,
        padding="max_length"
    )

In [23]:
train_tok = train_ds.map(tok, batched=True)
test_tok  = test_ds.map(tok, batched=True)

cols = ["input_ids", "attention_mask", "label"]
train_tok.set_format(type="torch", columns=cols)
test_tok.set_format(type="torch", columns=cols)

Map:   0%|          | 0/11324 [00:00<?, ? examples/s]

Map:   0%|          | 0/2831 [00:00<?, ? examples/s]

#### Метрики

In [24]:
acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

#### Обучение

In [25]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

args = TrainingArguments(
    output_dir="/content/rubert_tiny_toxic",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=10,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()
trainer.evaluate()

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider trai

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.272230,0.247655,0.907453,0.856044
2,0.204168,0.242621,0.913458,0.865753
3,0.165924,0.232893,0.911339,0.871348
4,0.157139,0.234113,0.912045,0.868602
5,0.142801,0.250197,0.912752,0.868966
6,0.117441,0.256184,0.915578,0.874672
7,0.116075,0.299263,0.912045,0.863711
8,0.092683,0.300378,0.911339,0.863513
9,0.087974,0.300548,0.910986,0.864370


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.

{'eval_loss': 0.25642129778862,
 'eval_accuracy': 0.9152243023666549,
 'eval_f1': 0.8740818467995802,
 'eval_runtime': 1.0915,
 'eval_samples_per_second': 2593.575,
 'eval_steps_per_second': 41.226,
 'epoch': 9.0}

#### Валидация

In [27]:
model.eval()
model.to("cpu")

all_probs = []
all_labels = []

with torch.no_grad():
    for batch in torch.utils.data.DataLoader(test_tok, batch_size=128):
        labels = batch["label"].numpy()
        batch = {k: v for k, v in batch.items() if k != "label"}
        out = model(**batch)
        probs = F.softmax(out.logits, dim=-1)[:, 1].cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels)

proba_tr = np.concatenate(all_probs)
y_true = np.concatenate(all_labels)

metrics_rubert = eval_binary(y_true, proba_tr, title="ruBERT-tiny2")

ruBERT-tiny2 {'acc': 0.9152243023666549, 'f1': 0.8740818467995802, 'roc_auc': np.float64(0.9674117838486874)}
              precision    recall  f1-score   support

           0     0.9401    0.9321    0.9361      1886
           1     0.8668    0.8815    0.8741       945

    accuracy                         0.9152      2831
   macro avg     0.9035    0.9068    0.9051      2831
weighted avg     0.9156    0.9152    0.9154      2831



#### Попробуем динамическое квантование

In [28]:
model_fp32 = model
model_int8 = torch.quantization.quantize_dynamic(
    model_fp32,
    {torch.nn.Linear},
    dtype=torch.qint8
)
model_int8.eval()

/tmp/ipykernel_178/34404562.py:2: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = torch.quantization.quantize_dynamic(


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(83828, 312, padding_idx=0)
      (position_embeddings): Embedding(2048, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): DynamicQuantizedLinear(in_features=312, out_features=312, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
              (key): DynamicQuantizedLinear(in_features=312, out_features=312, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
              (value): DynamicQuantizedLinear(in_features=312, out_features=312, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
              (dropout): Dropout(p=0.1, inplace=False)
            )

In [29]:
all_probs = []
all_labels = []

with torch.no_grad():
    for batch in torch.utils.data.DataLoader(test_tok, batch_size=128):
        labels = batch["label"].numpy()
        batch = {k: v for k, v in batch.items() if k != "label"}
        out = model_int8(**batch)
        probs = F.softmax(out.logits, dim=-1)[:, 1].cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels)

proba_tr = np.concatenate(all_probs)
y_true = np.concatenate(all_labels)

metrics_rubert_q = eval_binary(y_true, proba_tr, title="ruBERT-tiny2-quantized")

ruBERT-tiny2-quantized {'acc': 0.9141646061462381, 'f1': 0.8744186046511628, 'roc_auc': np.float64(0.9674639644947174)}
              precision    recall  f1-score   support

           0     0.9462    0.9236    0.9348      1886
           1     0.8545    0.8952    0.8744       945

    accuracy                         0.9142      2831
   macro avg     0.9004    0.9094    0.9046      2831
weighted avg     0.9156    0.9142    0.9146      2831



In [30]:
pd.DataFrame([
    {"model": "LogReg+TFIDF", **metrics_lr},
    {"model": "ruBERT-tiny2", **metrics_rubert},
    {"model": "ruBERT-tiny2-quantized", **metrics_rubert_q},
])

,model,acc,f1,roc_auc
0,LogReg+TFIDF,0.876369,0.798619,0.937332
1,ruBERT-tiny2,0.915224,0.874082,0.967412
2,ruBERT-tiny2-quantized,0.914165,0.874419,0.967464


### Проверим latency

In [31]:
def latency_benchmark(fn, n_warmup=30, n_runs=200):
    for _ in range(n_warmup):
        fn()

    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn()
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000.0)

    times = np.array(times)
    return {
        "p50_ms": float(np.percentile(times, 50)),
        "p95_ms": float(np.percentile(times, 95)),
        "p99_ms": float(np.percentile(times, 99)),
        "mean_ms": float(times.mean()),
    }

In [32]:
sample_text = "Ты вообще ничего не понимаешь, иди отсюда!"

#### Логрег

In [33]:
def lr_infer_once():
    x = preprocess_one_cached(sample_text)
    _ = lr_tfidf.predict_proba([x])[0, 1]

lat_lr = latency_benchmark(lr_infer_once)
lat_lr

{'p50_ms': 0.8805004999885568,
 'p95_ms': 0.9697636999618451,
 'p99_ms': 1.0428111499987835,
 'mean_ms': 0.877262004999011}

#### Трансформеры

In [34]:
torch.set_num_threads(1)  # часто улучшает latency на небольших батчах; можно подобрать 1/2/4

def tr_infer_once(model_):
    enc = tokenizer(
        sample_text,
        truncation=True,
        max_length=max_length,
        padding="max_length",
        return_tensors="pt"
    )
    with torch.no_grad():
        out = model_(**enc)
        _ = F.softmax(out.logits, dim=-1)[0, 1].item()

lat_tr_fp32 = latency_benchmark(lambda: tr_infer_once(model_fp32))
lat_tr_int8 = latency_benchmark(lambda: tr_infer_once(model_int8))

lat_tr_fp32, lat_tr_int8

({'p50_ms': 11.285367500022403,
  'p95_ms': 17.77479145003155,
  'p99_ms': 19.43035696002197,
  'mean_ms': 12.27110758999828},
 {'p50_ms': 16.053696499994885,
  'p95_ms': 19.551037249993893,
  'p99_ms': 23.63378070997555,
  'mean_ms': 15.547291664998397})